In [1]:
%run 00_functions.ipynb

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_df = pd.read_csv("../data/lstm/raw/bitcoin_historical_data.csv")

raw_df.head()

,date,open,high,low,close,volume
0,01/01/2018,13850.490234,13921.530273,12877.669922,13444.879883,78425
1,01/02/2018,13444.879883,15306.129883,12934.160156,14754.129883,137732
2,01/03/2018,14754.089844,15435.009766,14579.709961,15156.620117,106543
3,01/04/2018,15156.490234,15408.660156,14244.669922,15180.080078,110969
4,01/05/2018,15180.080078,17126.949219,14832.360352,16954.779297,141960


In [3]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3208 entries, 0 to 3207
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    3208 non-null   str    
 1   open    3208 non-null   float64
 2   high    3208 non-null   float64
 3   low     3208 non-null   float64
 4   close   3208 non-null   float64
 5   volume  3208 non-null   int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 150.5 KB


In [4]:
print("Number of rows:", raw_df.shape[0])
print("Number of columns:", raw_df.shape[1])

Number of rows: 3208
Number of columns: 6


In [5]:
print(raw_df.columns.tolist())

['date', 'open', 'high', 'low', 'close', 'volume']


In [6]:
print("Missing values:")
print(raw_df.isnull().sum())

Missing values:
date      0
open      0
high      0
low       0
close     0
volume    0
dtype: int64


In [7]:
print(
    "Duplicate dates:",
    raw_df["date"].duplicated().sum()
)

Duplicate dates: 0


In [8]:
raw_df["date"] = pd.to_datetime(
    raw_df["date"],
    format="%m/%d/%Y"
)

raw_df.head()

,date,open,high,low,close,volume
0,2018-01-01,13850.490234,13921.530273,12877.669922,13444.879883,78425
1,2018-01-02,13444.879883,15306.129883,12934.160156,14754.129883,137732
2,2018-01-03,14754.089844,15435.009766,14579.709961,15156.620117,106543
3,2018-01-04,15156.490234,15408.660156,14244.669922,15180.080078,110969
4,2018-01-05,15180.080078,17126.949219,14832.360352,16954.779297,141960


In [9]:
print("First date:", raw_df["date"].min())
print("Last date:", raw_df["date"].max())

First date: 2018-01-01 00:00:00
Last date: 2026-10-13 00:00:00


In [10]:
print(
    "Chronological order:",
    raw_df["date"].is_monotonic_increasing
)

Chronological order: True


In [11]:
raw_df = raw_df.sort_values(
    by="date"
).reset_index(drop=True)

In [12]:
print(
    "Chronological order:",
    raw_df["date"].is_monotonic_increasing
)

Chronological order: True


In [13]:
print(
    "High < Open:",
    (raw_df["high"] < raw_df["open"]).sum()
)

print(
    "High < Close:",
    (raw_df["high"] < raw_df["close"]).sum()
)

print(
    "Low > Open:",
    (raw_df["low"] > raw_df["open"]).sum()
)

print(
    "Low > Close:",
    (raw_df["low"] > raw_df["close"]).sum()
)

High < Open: 0
High < Close: 0
Low > Open: 0
Low > Close: 0


In [14]:
print(
    "Non-positive Open:",
    (raw_df["open"] <= 0).sum()
)

print(
    "Non-positive High:",
    (raw_df["high"] <= 0).sum()
)

print(
    "Non-positive Low:",
    (raw_df["low"] <= 0).sum()
)

print(
    "Non-positive Close:",
    (raw_df["close"] <= 0).sum()
)

Non-positive Open: 0
Non-positive High: 0
Non-positive Low: 0
Non-positive Close: 0


In [15]:
raw_df.describe()

,date,open,high,low,close,volume
count,3208,3208.000000,3208.000000,3208.000000,3208.000000,3.208000e+03
mean,2022-05-23 12:00:00,40480.921055,41274.010802,39639.408388,40503.015840,1.923409e+07
min,2018-01-01 00:00:00,3228.637939,3282.261963,3176.994385,3228.681885,2.640000e+02
25%,2020-03-12 18:00:00,9944.731201,10251.157959,9655.967773,9944.672363,6.022775e+04
50%,2022-05-23 12:00:00,30331.793945,30849.319336,29805.969727,30364.297852,1.100670e+05
75%,2024-08-02 06:00:00,63934.129883,64913.707031,62806.373047,63962.054688,3.745392e+05
max,2026-10-13 00:00:00,124687.515625,126185.992188,123144.562500,124725.109375,4.468697e+09
std,NaN,32553.650621,33054.593052,32025.380578,32559.948740,1.929496e+08


In [16]:
print("Final shape:", raw_df.shape)
print("Missing values:", raw_df.isnull().sum().sum())
print("Duplicate dates:", raw_df["date"].duplicated().sum())
print(
    "Chronological order:",
    raw_df["date"].is_monotonic_increasing
)
print("Missing Volume:", raw_df["volume"].isnull().sum())
print("Non-positive Volume:", (raw_df["volume"] <= 0).sum())

Final shape: (3208, 6)
Missing values: 0
Duplicate dates: 0
Chronological order: True
Missing Volume: 0
Non-positive Volume: 0


In [17]:
raw_df.to_csv("../data/lstm/processed/bitcoin_cleaned_data.csv",index=False)